In [84]:
import os
import time

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader  
from langchain_text_splitters import RecursiveCharacterTextSplitter  
# from langchain_pinecone import PineconeVectorStore
# from pinecone import Pinecone, ServerlessSpec
from langchain_community.vectorstores import Pinecone
from langchain_community.llms import CTransformers  # ← changed
from langchain_core.runnables import RunnablePassthrough

In [50]:

# from langchain_community.chains import RetrievalQA

In [51]:
#Extract data from the PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [39]:
extracted_data = load_pdf("../data/")

In [40]:
# extracted_data

In [52]:
# Create text chunks

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chuks = text_splitter.split_documents(extracted_data)
    
    return text_chuks
    

In [42]:
text_chunks = text_split(extracted_data)
print(f'length of text chunks: {len(text_chunks)}')

length of text chunks: 5860


In [43]:
# Create embeddings and store in Pinecone
def download_embedding_model():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [44]:
embeddings = download_embedding_model()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5576.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [61]:
from pinecone import Pinecone, ServerlessSpec
import os

api_key = os.getenv("PINECONE_API_KEY")
index_name="medical-chatbot"

# Initialize Pinecone
pc = Pinecone(api_key=api_key)

In [67]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,  # ← Document objects, not raw strings
    embedding=embeddings,
    index_name=index_name
)

In [68]:
docsearch

In [75]:
#If we already have an index we can load it like this
docsearch=PineconeVectorStore.from_existing_index(index_name, embeddings)

query = "What are Allergies"

docs=docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='97f626a1-12c2-4090-8c3e-e3d973ca3feb', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 135, 'page_label': '136', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': '../data/Medical_book.pdf', 'total_pages': 637}, page_content='Purpose\nAllergy is a reaction of the immune system. Nor-\nmally, the immune system responds to foreign microor-\nganisms and particles, like pollen or dust, by producing\nspecific proteins called antibodies that are capable of\nbinding to identifying molecules, or antigens, on the\nforeign organisms. This reaction between antibody and\nantigen sets off a series of reactions designed to protect\nthe body from infection. Sometimes, this same series of'), Document(id='e116e01d-772a-47e7-a6a1-5c7ddc2a7893', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 129, 'page_label': '130', 'producer': 'PDFlib

In [76]:
prompt_template="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [77]:
PROMPT=PromptTemplate(template=prompt_template, input_variables=["context", "question"])
chain_type_kwargs={"prompt": PROMPT}

In [78]:
llm=CTransformers(model="../model/llama-2-7b-chat.ggmlv3.q4_0.bin",
                  model_type="llama",
                  config={'max_new_tokens':512,
                          'temperature':0.8})

In [79]:
retriever = docsearch.as_retriever(search_kwargs={"k": 2})

In [83]:
template = """You are a medical expert. Answer the question based on the context.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)


In [85]:
chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)


In [ ]:
user_question = "What causes diabetes?"
print(f"Question: {user_question}\n")
print("⏳ Waiting for response (local model, takes 5-30 seconds)...\n")

result = chain.invoke(user_question)

print("✅ Answer:")
print(result)

Question: What causes diabetes?

⏳ Waiting for response (local model, takes 5-30 seconds)...



Number of tokens (678) exceeded maximum context length (512).
